# RF-DETR-Seg Training on Google Colab (Vineyard AI Challenge)

Train **RF-DETR-Seg Large** (`RFDETRSegLarge`, 704x704) with DINOv2 vision transformer backbone on Google Colab with GPU acceleration (T4 / A100 / L4).

### Dataset: EU ICAERUS Vine-Specific Segmentation Dataset
- **59 training tiles** (795 annotated vine canopy polygons)
- **26 validation tiles** (298 annotated vine canopy polygons)
- Formatted in COCO Polygon Segmentation (`_annotations.coco.json`).

## 1. Environment & GPU Setup

In [ ]:
!nvidia-smi
!pip install --upgrade pip
!pip install "rfdetr[train,loggers]" supervision


## 2. Prepare Dataset
RF-DETR uses the **COCO Segmentation format**.
Upload `dataset_coco.zip` containing:
```
dataset/
  ├── train/
  │     └── _annotations.coco.json
  ├── valid/
  │     └── _annotations.coco.json
  └── test/
        └── _annotations.coco.json
```

In [ ]:
# Unzip COCO dataset
!unzip -q dataset_coco.zip -d /content/dataset

## 3. Train RF-DETR-Seg (Recommended: Large / Medium)

RF-DETR-L (RFDETRSegLarge) uses the same lightweight DINOv2-Small backbone (~33.9M params vs 30.5M for Nano), but runs at **704x704 resolution** (+8% higher AP and much sharper boundaries on thin aerial vine canopies).

In [ ]:
from rfdetr import RFDETRSegLarge, RFDETRSegMedium, RFDETRSegNano

# Recommended for high-resolution aerial canopies: RFDETRSegLarge (704x704, 33.9M params)
model = RFDETRSegLarge()
# Alternatively: model = RFDETRSegMedium()  # 576x576, 33.7M params
# Alternatively: model = RFDETRSegNano()    # 384x384, 30.5M params

# Train on custom dataset
model.train(
    dataset_dir="/content/dataset",
    epochs=35,
    batch_size=4,       # 4 works comfortably on free Colab T4 GPU (15GB VRAM)
    device="cuda",
    output_dir="/content/rfdetr_output",
)


## 4. Test Model Predictions

In [ ]:
import supervision as sv
from PIL import Image
import glob

test_images = glob.glob("/content/dataset/valid/*.jpg") + glob.glob("/content/dataset/valid/*.png")
if test_images:
    sample_img = test_images[0]
    detections = model.predict(sample_img, threshold=0.35)
    print(f"Detected {len(detections)} canopy instances in {sample_img}")
    
    # Convert to RGB (in case image is 4-channel RGBA)
    img = Image.open(sample_img).convert("RGB")
    annotated = sv.MaskAnnotator().annotate(img, detections)
    annotated.save("/content/test_prediction.png")
    print("Saved prediction preview to /content/test_prediction.png")


## 5. Download the Trained Model Weights

In [ ]:
from google.colab import files
import os, glob

print("Contents of /content/rfdetr_output:")
!ls -lh /content/rfdetr_output

checkpoints = sorted(glob.glob("/content/rfdetr_output/**/checkpoint_best_*.pth", recursive=True) + glob.glob("/content/rfdetr_output/**/best.pt", recursive=True))
if checkpoints:
    print(f"Downloading best checkpoint: {checkpoints[-1]}...")
    files.download(checkpoints[-1])
else:
    all_ckpts = glob.glob("/content/rfdetr_output/**/*.pth", recursive=True) + glob.glob("/content/rfdetr_output/**/*.ckpt", recursive=True)
    if all_ckpts:
        print(f"Downloading {all_ckpts[0]}...")
        files.download(all_ckpts[0])
    else:
        print("Please check the output directory for checkpoint files.")
